In [ ]:
# Import the required PySpark modules
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from pyspark.sql.functions import window, avg

# ---------------------------------------------------------
# Step 1: Initialize Spark Session (FIXED FOR LOCAL HIVE)
### Overriding the default Hive connection to use a local embedded 
### database so it doesn't crash looking for 'cluster_util_db'
# ---------------------------------------------------------

In [ ]:
spark = SparkSession.builder \
    .appName("Mohammed_Lab2_Temperature_Streaming") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "file:///data/spark-warehouse") \
    .config("javax.jdo.option.ConnectionURL", "jdbc:derby:;databaseName=/data/metastore_db;create=true") \
    .config("spark.sql.shuffle.partitions", "4") \
    .enableHiveSupport() \
    .getOrCreate()

# ---------------------------------------------------------
# Step 2: Define the JSON Schema
# ---------------------------------------------------------

In [3]:
json_schema = StructType([
    StructField("event_timestamp", TimestampType(), True),
    StructField("country", StringType(), True),
    StructField("temperature", DoubleType(), True)
])

# ---------------------------------------------------------
# Step 3: Read the Stream from the JSON Directory
# ---------------------------------------------------------

In [4]:
streaming_df = spark.readStream \
    .schema(json_schema) \
    .option("maxFilesPerTrigger", 1) \
    .json("file:///data/json/")

# ---------------------------------------------------------
# Step 4: Windowing & Watermarking (Transformations)
# ---------------------------------------------------------

In [5]:
# 1. withWatermark: Discards data late by more than 10 minutes.
# 2. groupBy: Groups by a 15-minute time window and the country.
# 3. agg: Calculates the average temperature for that group.
agg_df = streaming_df \
    .withWatermark("event_timestamp", "10 minutes") \
    .groupBy(
        window("event_timestamp", "15 minutes"),
        "country"
    ) \
    .agg(avg("temperature").alias("avg_temperature"))

# ---------------------------------------------------------
# Step 5: Define a function to write to multiple locations
# ---------------------------------------------------------

In [ ]:
# foreachBatch allows us to take the micro-batch generated by the 
# stream and write it to as many places as we want simultaneously.
def write_to_multiple_sinks(batch_df, batch_id):
    
    # Location 1: Write to the filesystem as Parquet files
    batch_df.write \
        .mode("append") \
        .parquet("file:///data/output/filesystem_avg_temps/")
        
    # Location 2: Write to a Hive Table
    batch_df.write \
        .mode("append") \
        .saveAsTable("hive_country_avg_temp")

# ---------------------------------------------------------
# Step 6: Start the Stream with the Correct Output Mode
# ---------------------------------------------------------

In [ ]:
# Append mode is strictly required when writing windowed 
# aggregations to file-based sinks.
query = agg_df.writeStream \
    .outputMode("append") \
    .option("checkpointLocation", "file:///data/checkpoint/") \
    .foreachBatch(write_to_multiple_sinks) \
    .start()

# Keep the streaming application running
query.awaitTermination()

KeyboardInterrupt: 

# ---------------------------------------------------------
# Verification: Read from both output sinks
# ---------------------------------------------------------

In [8]:
# 1. Read and display data from the Filesystem (Parquet files)
print("==================================================")
print("OUTPUT SINK 1: Filesystem (Parquet)")
print("==================================================")
try:
    parquet_df = spark.read.parquet("file:///data/output/filesystem_avg_temps/")
    # Order by window and country to make it easy to read
    parquet_df.orderBy("window", "country").show(truncate=False)
except Exception as e:
    print(f"Filesystem check failed. Did you drop enough JSON files to pass the 10-minute watermark?\nError: {e}")

OUTPUT SINK 1: Filesystem (Parquet)
+------------------------------------------+---------+---------------+
|window                                    |country  |avg_temperature|
+------------------------------------------+---------+---------------+
|[2024-01-15 10:00:00, 2024-01-15 10:15:00]|Japan    |8.7            |
|[2024-01-15 10:00:00, 2024-01-15 10:15:00]|UK       |5.2            |
|[2024-01-15 10:00:00, 2024-01-15 10:15:00]|USA      |12.5           |
|[2024-01-15 10:15:00, 2024-01-15 10:30:00]|Brazil   |28.9           |
|[2024-01-15 10:15:00, 2024-01-15 10:30:00]|France   |6.8            |
|[2024-01-15 10:15:00, 2024-01-15 10:30:00]|Germany  |3.4            |
|[2024-01-15 10:15:00, 2024-01-15 10:30:00]|India    |22.5           |
|[2024-01-15 10:15:00, 2024-01-15 10:30:00]|USA      |13.1           |
|[2024-01-15 10:30:00, 2024-01-15 10:45:00]|Australia|31.2           |
|[2024-01-15 10:30:00, 2024-01-15 10:45:00]|Germany  |3.9            |
|[2024-01-15 10:30:00, 2024-01-15 10:45:0

In [9]:
# 2. Read and display data from the local Hive Table
print("==================================================")
print("OUTPUT SINK 2: Hive Table")
print("==================================================")
try:
    hive_df = spark.sql("SELECT * FROM hive_country_avg_temp ORDER BY window, country")
    hive_df.show(truncate=False)
except Exception as e:
    print(f"Hive table check failed.\nError: {e}")

OUTPUT SINK 2: Hive Table
+------------------------------------------+---------+---------------+
|window                                    |country  |avg_temperature|
+------------------------------------------+---------+---------------+
|[2024-01-15 10:00:00, 2024-01-15 10:15:00]|Japan    |8.7            |
|[2024-01-15 10:00:00, 2024-01-15 10:15:00]|UK       |5.2            |
|[2024-01-15 10:00:00, 2024-01-15 10:15:00]|USA      |12.5           |
|[2024-01-15 10:15:00, 2024-01-15 10:30:00]|Brazil   |28.9           |
|[2024-01-15 10:15:00, 2024-01-15 10:30:00]|France   |6.8            |
|[2024-01-15 10:15:00, 2024-01-15 10:30:00]|Germany  |3.4            |
|[2024-01-15 10:15:00, 2024-01-15 10:30:00]|India    |22.5           |
|[2024-01-15 10:15:00, 2024-01-15 10:30:00]|USA      |13.1           |
|[2024-01-15 10:30:00, 2024-01-15 10:45:00]|Australia|31.2           |
|[2024-01-15 10:30:00, 2024-01-15 10:45:00]|Germany  |3.9            |
|[2024-01-15 10:30:00, 2024-01-15 10:45:00]|Japan  